In [1]:
import torch
from torch.masked import masked_tensor, as_masked_tensor

### Fake data setup

In [2]:
n_clouds = 2
n_particles = 3
n_features = 4

In [3]:
torch.manual_seed(42)
x = torch.randn(n_clouds, n_particles, n_features)
n_positive_mask_vals = torch.round(n_particles * torch.rand(2)).long()
n_positive_mask_vals

tensor([2, 1])

In [4]:
masks = [torch.concat((torch.ones(int(n_ones)), torch.zeros(n_particles - int(n_ones))), dim=0) for n_ones in n_positive_mask_vals.numpy()]
masks = torch.stack(masks, dim=0)
masks = masks.unsqueeze(-1).expand(-1, -1, x.shape[-1])
masks

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.],
         [0., 0., 0., 0.]],

        [[1., 1., 1., 1.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])

In [34]:
masks.shape

torch.Size([2, 3, 4])

### Production code

In [5]:
x_masked = masks * x

In [6]:
x_i = x_masked.unsqueeze(-2) # second-last dimension - N
x_j = x_masked.unsqueeze(-3) # third-last dimension - B
diff = x_i - x_j
diff.shape

torch.Size([2, 3, 3, 4])

In [7]:
diff

tensor([[[[ 0.0000,  0.0000,  0.0000,  0.0000],
          [ 1.2485,  2.7218,  0.9438, -0.5009],
          [ 1.9269,  1.4873,  0.9007, -2.1055]],

         [[-1.2485, -2.7218, -0.9438,  0.5009],
          [ 0.0000,  0.0000,  0.0000,  0.0000],
          [ 0.6784, -1.2345, -0.0431, -1.6047]],

         [[-1.9269, -1.4873, -0.9007,  2.1055],
          [-0.6784,  1.2345,  0.0431,  1.6047],
          [ 0.0000,  0.0000,  0.0000,  0.0000]]],


        [[[ 0.0000,  0.0000,  0.0000,  0.0000],
          [-1.1109,  0.0915, -2.3169, -0.2168],
          [-1.1109,  0.0915, -2.3169, -0.2168]],

         [[ 1.1109, -0.0915,  2.3169,  0.2168],
          [ 0.0000,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000, -0.0000]],

         [[ 1.1109, -0.0915,  2.3169,  0.2168],
          [ 0.0000,  0.0000, -0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  0.0000]]]])

In [12]:
masks.shape

torch.Size([2, 3, 4])

We need to expand masks along its second-from-last dimension.

In [20]:
expanded_mask = masks.unsqueeze(-3).expand(-1, n_particles, -1, -1)
expanded_mask

tensor([[[[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [0., 0., 0., 0.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [0., 0., 0., 0.]],

         [[1., 1., 1., 1.],
          [1., 1., 1., 1.],
          [0., 0., 0., 0.]]],


        [[[1., 1., 1., 1.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],

         [[1., 1., 1., 1.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],

         [[1., 1., 1., 1.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]])

In [35]:
x.shape

torch.Size([2, 3, 4])

In [22]:
diff_masked = diff * expanded_mask
diff_masked.shape

torch.Size([2, 3, 3, 4])

In [23]:
from util.minkowski_utils import normsq4, dotsq4

In [ ]:
norms = normsq4(diff_masked)
dots = dotsq4(x_i, x_j)

In [33]:
norms

tensor([[[ 0.0000, -6.9912,  0.0000],
         [-6.9912,  0.0000,  0.0000],
         [-3.7435, -3.6407,  0.0000]],

        [[ 0.0000,  0.0000,  0.0000],
         [-4.1894,  0.0000,  0.0000],
         [-4.1894,  0.0000,  0.0000]]])

In [ ]:
def message_passing(norms, dots, diffs):
    inp = torch.stack([norms, dots], dim=-1)  # Concatenate along feature dimension
    # print(f"{inp.shape=}")
    out = self.phi_e(inp)
    # print(f"phi_e(norms, dots) = {out.shape}")
    out = out * self.phi_m(out)
    
    return out